设置默认语言为中文，包括注释、文字说明和总结、代码中的print，全都使用中文！

In [ ]:
%run /content/drive/MyDrive/etrain/colab_init.py

!pip install faiss-cpu
#import matplotlib.pyplot as plt已经执行不要他妈重复导入！！！！！！！！！

正在配置中文字体...
已成功切换至中文字体: Noto Sans CJK JP
正在安装项目依赖...
环境初始化完成！
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 17.2 MB/s eta 0:00:00


In [ ]:
from utils.pl_exp import PolarsExp as pe

def get_indicators(df):
    # 计算 MACD 和不同周期的随机指标
    return df.with_columns(
        *pe.macd(),
        *pe.stoch(k_period=29, alias_suffix=True),
        *pe.stoch(k_period=69, alias_suffix=True),
    )

In [ ]:
import os
import polars as pl
import numpy as np
from utils.pl_data import get_pl_kline_df

# 1. 运行初始化脚本


# 2. 配置路径与参数
token = 'ETH'
base_period = 1
target_period = 15
data_dir = '/content/drive/MyDrive/kline_data'

# 3. 加载 Polars 数据
print(f'正在从 {data_dir} 加载 {token} 数据...')
pl_df_base = get_pl_kline_df(data_dir, token, period=base_period)

# 4. 添加目标标签: 15分钟后的价格是否高于当前价格
# 注意：shift(-15) 将未来的价格移动到当前行进行比较
pl_df_base = pl_df_base.with_columns(
    target = (pl.col('close').shift(-target_period) > pl.col('close'))
)

# 5. 验证数据
print(f'\n数据加载完成！')
print(f'数据总量: {len(pl_df_base)}')
print(f'目标标签分布:\n{pl_df_base.select(pl.col("target").value_counts())}')
print(pl_df_base.head())

正在从 /content/drive/MyDrive/kline_data 加载 ETH 数据...
从本地读取数据耗时: 15.35 秒

数据加载完成！
数据总量: 2980800
目标标签分布:
shape: (3, 1)
┌─────────────────┐
│ target          │
│ ---             │
│ struct[2]       │
╞═════════════════╡
│ {false,1480949} │
│ {true,1499836}  │
│ {null,15}       │
└─────────────────┘
shape: (5, 18)
┌────────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬────────┐
│ ts         ┆ open       ┆ high      ┆ low       ┆ … ┆ scvd_h    ┆ scvd_l    ┆ scvd_o    ┆ target │
│ ---        ┆ ---        ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---    │
│ datetime[m ┆ f32        ┆ f32       ┆ f32       ┆   ┆ f32       ┆ f32       ┆ f32       ┆ bool   │
│ s]         ┆            ┆           ┆           ┆   ┆           ┆           ┆           ┆        │
╞════════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪════════╡
│ 2020-01-01 ┆ 129.119995 ┆ 129.11999 ┆ 128.91000 ┆ … ┆ -115.7715 ┆ -115.7715 ┆ -115

## ACE 策略重构区
此区域之后的单元格已被清理，准备进行全新的生产级重构。

In [ ]:
import numpy as np
import faiss
import polars as pl
import gc

# 1. 核心配置定义
HYBRID_CONFIG = {
    'v_short_10': {'window': 10, 'n_clusters': 100, 'repeat': 0},
    'short_15':   {'window': 15, 'n_clusters': 120, 'repeat': 10},
    'short_30':   {'window': 30, 'n_clusters': 150, 'repeat': 15},
    'mid_60':     {'window': 60, 'n_clusters': 300, 'repeat': 15},
    'long_120':    {'window': 120, 'n_clusters': 600, 'repeat': 15}
}

# 2. 生产级无泄露聚类引擎
def run_hybrid_multi_scale_clustering_leakproof(train_data, test_data, config, is_indicator=False, end_k=3):
    tr_ids_dict = {}
    te_ids_dict = {}

    for name, cfg in config.items():
        w, n_c, r_count = cfg['window'], cfg['n_clusters'], cfg['repeat']
        print(f"正在处理尺度 (无泄露版): {name}...")

        def get_feats(data_array):
            # 构建窗口特征
            shape = (len(data_array) - w + 1, w)
            strides = (data_array.strides[0], data_array.strides[0])
            windows = np.lib.stride_tricks.as_strided(data_array, shape=shape, strides=strides)

            if is_indicator:
                norm = windows / 100.0
            else:
                w_min, w_max = windows.min(1, keepdims=True), windows.max(1, keepdims=True)
                norm = (windows - w_min) / np.where((w_max - w_min) == 0, 1e-9, w_max - w_min)

            # 末端强化逻辑
            if r_count > 0:
                feat = np.hstack([norm] + [norm[:, -end_k:]] * r_count)
            else:
                feat = norm
            return np.ascontiguousarray(feat, dtype='float32')

        tr_feat = get_feats(train_data)
        te_feat = get_feats(test_data)

        # 严格限制：仅使用训练集特征拟合聚类中心
        km = faiss.Kmeans(tr_feat.shape[1], n_c, niter=40, gpu=True)
        km.train(tr_feat)

        _, tr_ids = km.index.search(tr_feat, 1)
        _, te_ids = km.index.search(te_feat, 1)

        # 填充起始空位对齐原数据索引
        tr_ids_dict[f'cluster_{name}'] = np.concatenate([np.full(w-1, -1, dtype=np.int32), tr_ids.flatten()])
        te_ids_dict[f'cluster_{name}'] = np.concatenate([np.full(w-1, -1, dtype=np.int32), te_ids.flatten()])

        del tr_feat, te_feat; gc.collect()

    return tr_ids_dict, te_ids_dict

# ACE 生产级核心逻辑 (已清理冗余实验)

In [ ]:
def report_real_performance(df, title):
    print(f'\n--- {title} ---')
    # 预处理：确保 target 是 i32，ace_score 是 f32，剔除无效值
    clean_df = df.filter(
        pl.col('target').is_not_null() &
        pl.col('ace_score').is_not_null() &
        pl.col('ace_score').is_finite()
    ).with_columns([
        pl.col('target').cast(pl.Boolean).cast(pl.Int32),
        pl.col('ace_score').cast(pl.Float32)
    ])

    total_test_rows = len(clean_df)
    if total_test_rows == 0:
        print("错误：没有有效数据可供分析。")
        return

    df_sorted = clean_df.with_columns(abs_s = pl.col('ace_score').abs()).sort('abs_s', descending=True)

    results = []
    for cov_rate in [0.01, 0.02, 0.05, 0.10, 0.25]:
        count = int(total_test_rows * cov_rate)
        if count > 0:
            sub = df_sorted.head(count)
            # 计算准确率：得分方向与 target 一致即为正确
            acc = sub.select(acc = pl.when(pl.col('ace_score') > 0).then(pl.col('target') == 1).otherwise(pl.col('target') == 0).mean()).item()
            results.append({
                '全局覆盖率': f'{cov_rate:.0%}',
                '准确率': f'{acc:.2%}',
                '信号总数': count,
                '日均信号': round(count / (total_test_rows/1440), 2)
            })

    if results:
        display(pl.DataFrame(results))
    else:
        print("数据不足。")

In [ ]:
def run_leakproof_v30_execution(pl_df, config):
    print('>>> 启动 v30: 零泄露 ACE 生产级重验证...')

    # 1. 物理时序分割
    split_idx = int(len(pl_df) * 0.8)
    train_df = pl_df.slice(0, split_idx)
    test_df = pl_df.slice(split_idx)

    # 提取序列
    tr_prices = train_df['close'].to_numpy().astype(np.float32)
    te_prices = test_df['close'].to_numpy().astype(np.float32)

    # 指标计算 (Stoch 29)
    all_ind = get_indicators(pl_df)
    tr_k = np.nan_to_num(all_ind.slice(0, split_idx)['stoch_k_29_3'].to_numpy(), nan=50.0).astype(np.float32)
    te_k = np.nan_to_num(all_ind.slice(split_idx)['stoch_k_29_3'].to_numpy(), nan=50.0).astype(np.float32)

    # 2. 执行零泄露聚类
    print("--- 价格序列聚类 ---")
    tr_p_ids, te_p_ids = run_hybrid_multi_scale_clustering_leakproof(tr_prices, te_prices, config, is_indicator=False)
    print("--- 指标序列聚类 ---")
    tr_s_ids, te_s_ids = run_hybrid_multi_scale_clustering_leakproof(tr_k, te_k, config, is_indicator=True)

    # 3. 合并特征
    train_final = train_df.with_columns([pl.Series(k, v) for k, v in {**tr_p_ids, **tr_s_ids}.items()])
    test_final = test_df.with_columns([pl.Series(k, v) for k, v in {**te_p_ids, **te_s_ids}.items()])

    # 4. 仅在训练集计算每个簇的 Edge (胜率 - 0.5)
    cluster_cols = [c for c in train_final.columns if 'cluster_' in c]
    for col in cluster_cols:
        mapping = train_final.filter(pl.col(col) != -1).group_by(col).agg(
            (pl.col('target').mean() - 0.5).alias(f'edge_{col}')
        )
        test_final = test_final.join(mapping, on=col, how='left')

    # 5. 计算共振总分
    edge_cols = [c for c in test_final.columns if 'edge_' in c]
    test_final = test_final.with_columns(
        ace_score = pl.sum_horizontal([pl.col(c).fill_null(0) for c in edge_cols])
    )

    # 6. 输出报告
    report_real_performance(test_final, "v30_ZeroLeak_Final_Report")
    return test_final

# 触发执行
final_scored_v30 = run_leakproof_v30_execution(pl_df_base, HYBRID_CONFIG)

>>> 启动 v30: 零泄露 ACE 生产级重验证...
--- 价格序列聚类 ---
正在处理尺度 (无泄露版): v_short_10...
正在处理尺度 (无泄露版): short_15...
正在处理尺度 (无泄露版): short_30...
正在处理尺度 (无泄露版): mid_60...
正在处理尺度 (无泄露版): long_120...
--- 指标序列聚类 ---
正在处理尺度 (无泄露版): v_short_10...
正在处理尺度 (无泄露版): short_15...
正在处理尺度 (无泄露版): short_30...
正在处理尺度 (无泄露版): mid_60...
正在处理尺度 (无泄露版): long_120...

--- v30_ZeroLeak_Final_Report ---


全局覆盖率,准确率,信号总数,日均信号
str,str,i64,f64
"""1%""","""59.57%""",5961,14.4
"""2%""","""57.99%""",11922,28.8
"""5%""","""56.67%""",29807,72.0
"""10%""","""55.84%""",59614,144.0
"""25%""","""54.69%""",149036,360.0


### 信号方向分布深度分析
我们将分析在 10% 覆盖率下，看涨（正分）和看跌（负分）信号的具体表现。

In [ ]:
import polars as pl

def analyze_signal_directions(df, cov_rate=0.10):
    # 1. 数据清洗与预处理
    clean_df = df.filter(
        pl.col('target').is_not_null() &
        pl.col('ace_score').is_not_null() &
        pl.col('ace_score').is_finite()
    ).with_columns([
        pl.col('target').cast(pl.Boolean).cast(pl.Int32),
        pl.col('ace_score').cast(pl.Float32)
    ])

    # 2. 按得分绝对值排序，取前 10% 覆盖率的样本
    total_count = len(clean_df)
    top_count = int(total_count * cov_rate)
    df_sorted = clean_df.with_columns(abs_s = pl.col('ace_score').abs()).sort('abs_s', descending=True)
    top_df = df_sorted.head(top_count)

    # 3. 分别统计正分和负分的表现
    # 看涨信号 (ace_score > 0)
    long_signals = top_df.filter(pl.col('ace_score') > 0)
    long_acc = long_signals.select((pl.col('target') == 1).mean()).item() if len(long_signals) > 0 else 0

    # 看跌信号 (ace_score < 0)
    short_signals = top_df.filter(pl.col('ace_score') < 0)
    short_acc = short_signals.select((pl.col('target') == 0).mean()).item() if len(short_signals) > 0 else 0

    # 4. 构建汇总表
    summary = pl.DataFrame({
        "维度": ["看涨信号 (Long)", "看跌信号 (Short)", "总体 (Top 10%)"],
        "信号数量": [len(long_signals), len(short_signals), top_count],
        "占比": [f"{len(long_signals)/top_count:.2%}", f"{len(short_signals)/top_count:.2%}", "100%"],
        "准确率": [f"{long_acc:.2%}", f"{short_acc:.2%}", f"{top_df.select(acc = pl.when(pl.col('ace_score') > 0).then(pl.col('target') == 1).otherwise(pl.col('target') == 0).mean()).item():.2%}"]
    })

    print(f"\n--- 信号方向与准确率分布分析 (覆盖率: {cov_rate:.0%}) ---")
    display(summary)

# 执行分析
analyze_signal_directions(final_scored_v30, cov_rate=0.10)


--- 信号方向与准确率分布分析 (覆盖率: 10%) ---


维度,信号数量,占比,准确率
str,i64,str,str
"""看涨信号 (Long)""",39640,"""66.49%""","""56.04%"""
"""看跌信号 (Short)""",19974,"""33.51%""","""55.44%"""
"""总体 (Top 10%)""",59614,"""100%""","""55.84%"""


### 信号占比非对称性深度诊断
我们将检查训练集的标签基准分布，并分析各尺度下正负 Edge 的平均强度和分布情况。

In [ ]:
def diagnose_imbalance(df_scored, config):
    print("--- 策略非对称性诊断报告 ---")

    # 1. 检查测试集整体的基准胜率 (Baseline)
    baseline = df_scored.select(pl.col('target').cast(pl.Int32).mean()).item()
    print(f"测试集基准胜率 (Target=1 占比): {baseline:.2%}")

    # 2. 分析 ace_score 的分布特征
    stats = df_scored.select([
        pl.col('ace_score').filter(pl.col('ace_score') > 0).mean().alias('平均正向分值'),
        pl.col('ace_score').filter(pl.col('ace_score') < 0).mean().alias('平均负向分值'),
        pl.col('ace_score').skew().alias('得分偏度(Skewness)')
    ])
    display(stats)

    # 3. 逐尺度检查 Edge 强度
    edge_summary = []
    edge_cols = [c for c in df_scored.columns if 'edge_' in c]
    for col in edge_cols:
        pos_edge = df_scored.filter(pl.col(col) > 0).select(pl.col(col).mean()).item() or 0
        neg_edge = df_scored.filter(pl.col(col) < 0).select(pl.col(col).mean()).item() or 0
        edge_summary.append({
            "尺度": col.replace('edge_cluster_', ''),
            "平均正向Edge": round(pos_edge, 5),
            "平均负向Edge": round(neg_edge, 5),
            "强度比(正/负)": abs(pos_edge / neg_edge) if neg_edge != 0 else 0
        })

    print("\n各尺度 Edge 强度对比:")
    display(pl.DataFrame(edge_summary))

    print("\n提示: 如果基准胜率远超 50%，或者正向 Edge 强度显著高于负向，则解释了为何看涨信号更容易在排序中胜出。")

diagnose_imbalance(final_scored_v30, HYBRID_CONFIG)

--- 策略非对称性诊断报告 ---
测试集基准胜率 (Target=1 占比): 50.56%


平均正向分值,平均负向分值,得分偏度(Skewness)
f64,f64,f64
0.145878,-0.129395,0.033155



各尺度 Edge 强度对比:


尺度,平均正向Edge,平均负向Edge,强度比(正/负)
str,f64,f64,f64
"""v_short_10""",0.02846,-0.02582,1.102149
"""short_15""",0.02888,-0.02688,1.074395
"""short_30""",0.03029,-0.02676,1.131803
"""mid_60""",0.0317,-0.02853,1.111071
"""long_120""",0.03404,-0.0289,1.178054



提示: 如果基准胜率远超 50%，或者正向 Edge 强度显著高于负向，则解释了为何看涨信号更容易在排序中胜出。
